# Módulo A — Recorrido del flujo de pronóstico del gasto

Este notebook recorre, paso a paso, el mismo flujo que ejecuta `main.py`:
**ingesta → limpieza → serie temporal → EDA → modelos → evaluación**.

Útil para inspeccionar resultados intermedios de forma interactiva.

In [ ]:
# Permitir importar el paquete src/ al ejecutar el notebook desde notebooks/
import sys, os
sys.path.append(os.path.abspath('..'))

from src import ingesta, limpieza, serie_temporal, eda, modelos, evaluacion
modelos.fijar_semillas()

## 1. Ingesta y consolidación

In [ ]:
df_crudo, rep_ingesta = ingesta.consolidar(guardar=True)
print('Filas:', f"{rep_ingesta['filas_totales']:,}", '| Meses:', rep_ingesta['n_meses'])
print('Codificaciones:', rep_ingesta['codificaciones'])
df_crudo.head()

## 2. Limpieza y validación

In [ ]:
df_limpio, rep_limpieza = limpieza.limpiar(df_crudo, guardar=True)
print('Filas válidas:', f"{rep_limpieza['filas_finales']:,}")
print('Excluidas por estado:', rep_limpieza['ordenes_excluidas_por_estado'])

## 3. Serie temporal mensual

In [ ]:
serie_completa = serie_temporal.construir_serie_total(df_limpio)
deteccion = serie_temporal.detectar_meses_incompletos(serie_completa)
serie_total, serie_cat, _ = serie_temporal.construir_series(df_limpio, guardar=True)
print('Meses incompletos detectados:', deteccion['meses_incompletos'] or 'ninguno')
serie_total.tail()

## 4. EDA (genera figuras en outputs/figuras/)

In [ ]:
md_eda = eda.ejecutar_eda(df_limpio, serie_completa, deteccion, rep_ingesta, rep_limpieza)
print('Sección EDA generada (longitud):', len(md_eda), 'caracteres')

## 5-6. Modelos y evaluación

In [ ]:
md_modelos, resultados = evaluacion.ejecutar_evaluacion(serie_total)
print('Mejor modelo:', resultados['mejor_modelo'])
resultados['tabla_metricas'].round(2)

In [ ]:
# Pronóstico final
resultados['pronostico']